In [1]:
# # 02 - Validasi Sentimen 1.000 Data (3 penilai)
# Menggabungkan label sentimen dari 3 penilai pada 1.000 pasangan kalimat-aspek, membersihkan isian yang tidak valid (kosong / salah ketik), dan menampilkan sebaran label tiap penilai.
# 
# - Input : `validasi data/validasi_sentimen_1..3.xlsx` (sheet `label`)
# - Output: `hasil_validasi/label_penilai_1000.csv`, `hasil_validasi/label_tidak_valid.csv`


# ===== SEL 1 (salin ke satu sel notebook) =====
import os, re, itertools, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

FOLDER_DATA  = "validasi data"             # berkas validasi_*.xlsx dan aspek_kalimat.csv
FOLDER_HASIL = "hasil_validasi"   # semua keluaran disimpan di sini
os.makedirs(FOLDER_HASIL, exist_ok=True)

SEED = 42
LABEL = ["positif", "negatif", "netral"]
ASPEK = ["wifi", "harga", "suasana", "pelayanan"]
N_PENILAI = 3

def tafsir_kappa(k):
    """Skala Landis & Koch (1977)."""
    if k is None or np.isnan(k): return "tidak dapat dihitung"
    if k < 0:     return "buruk (di bawah kebetulan)"
    if k <= 0.20: return "sangat rendah"
    if k <= 0.40: return "rendah"
    if k <= 0.60: return "sedang"
    if k <= 0.80: return "baik (substantial)"
    return "sangat baik (almost perfect)"


# ===== SEL 2 (salin ke satu sel notebook) =====
dasar, mentah = None, {}
for p in range(1, N_PENILAI + 1):
    df = pd.read_excel(f"{FOLDER_DATA}/validasi_sentimen_{p}.xlsx", sheet_name="label")
    if dasar is None:
        dasar = df[["id_aspek_kalimat", "id_kalimat", "aspek", "kata_kunci", "kalimat_asli"]].copy()
    # pastikan urutan & isi baris sama antarpenilai
    assert (df["id_aspek_kalimat"].values == dasar["id_aspek_kalimat"].values).all(), f"urutan id berbeda pada penilai {p}"
    mentah[p] = df["label"]
print("Jumlah butir:", len(dasar))


# ## 1. Pembersihan label dan pelaporan isian tidak valid
# Isian di luar {positif, negatif, netral} **tidak ditebak**; dianggap kosong dan dilaporkan agar penilai memperbaikinya.


# ===== SEL 3 (salin ke satu sel notebook) =====
bersih, tidak_valid = {}, []
for p, s in mentah.items():
    t = s.astype("string").str.strip().str.lower()
    valid = t.isin(LABEL)
    bersih[f"penilai_{p}"] = t.where(valid).astype(object).where(valid, np.nan).values
    for i in np.where(~valid)[0]:
        tidak_valid.append({"penilai": p, "id_aspek_kalimat": dasar.iloc[i]["id_aspek_kalimat"],
                            "aspek": dasar.iloc[i]["aspek"], "isian_asli": s.iloc[i],
                            "kalimat_asli": dasar.iloc[i]["kalimat_asli"]})
tidak_valid = pd.DataFrame(tidak_valid, columns=["penilai", "id_aspek_kalimat", "aspek", "isian_asli", "kalimat_asli"])
print(f"Isian tidak valid / kosong: {len(tidak_valid)}")
tidak_valid


# ===== SEL 4 (salin ke satu sel notebook) =====
gabung = pd.concat([dasar.reset_index(drop=True), pd.DataFrame(bersih)], axis=1)
kolom = [f"penilai_{p}" for p in range(1, N_PENILAI + 1)]
gabung["n_penilai_valid"] = gabung[kolom].notna().sum(axis=1)
gabung.head()


# ## 2. Sebaran label tiap penilai


# ===== SEL 5 (salin ke satu sel notebook) =====
sebaran = pd.DataFrame({k: gabung[k].value_counts().reindex(LABEL).fillna(0).astype(int) for k in kolom})
sebaran.loc["kosong/tidak valid"] = gabung[kolom].isna().sum()
sebaran


# ===== SEL 6 (salin ke satu sel notebook) =====
# Sebaran per aspek dan penilai
pd.concat({k: pd.crosstab(gabung["aspek"], gabung[k]).reindex(columns=LABEL, fill_value=0) for k in kolom}, axis=1)


# ## 3. Simpan


# ===== SEL 7 (salin ke satu sel notebook) =====
gabung.to_csv(f"{FOLDER_HASIL}/label_penilai_1000.csv", index=False)
tidak_valid.to_csv(f"{FOLDER_HASIL}/label_tidak_valid.csv", index=False)
print("Tersimpan.")

Jumlah butir: 1000
Isian tidak valid / kosong: 0
Tersimpan.
